# Module 5: Multi-Agent Systems

**Day 4 — LangGraph Agents, Memory, HITL & MCP**

## What you will learn
- **Subgraphs**: independent graphs used as nodes inside a parent graph
- **Supervisor pattern**: one routing agent delegates to specialised sub-agents
- **Agent handoffs**: agents passing control to each other via state updates
- **Parallel agents**: fan-out to multiple agents simultaneously

## Why multi-agent?
- Tasks too large for a single context window
- Different specialisations (researcher, writer, coder, analyst)
- Parallelism: multiple agents working simultaneously = faster results

## Architecture
```
supervisor -> researcher -> supervisor -> writer -> supervisor -> finish
```


In [ ]:
import sys
sys.path.insert(0, '../src')
print('Path configured.')

## 1. Research Subgraph

A subgraph is an independent LangGraph that can be used as a node inside a larger graph.
This lets you compose complex systems from reusable building blocks.

In [ ]:
from day4.multi_agent import build_research_subgraph, build_writer_subgraph
from langchain_core.messages import HumanMessage, AIMessage

# Mock researcher
def mock_researcher(topic: str) -> str:
    data = {
        'india ai market': 'India AI market worth $7.8B in 2024. Key players: TCS, Infosys, Wipro. Govt AI Mission: $1.25B investment. 70,000 AI jobs created in 2024.',
        'default': f'Research on "{topic}": Found 5 key sources. Market growing 20% YoY. Technology advancing rapidly.'
    }
    return data.get(topic.lower(), data['default'])

research_graph = build_research_subgraph(mock_researcher)

result = research_graph.invoke({
    'messages': [],
    'topic': 'India AI market',
    'findings': None
})

print('Research subgraph result:')
print(f'  Topic: India AI market')
print(f'  Findings: {result["findings"][:100]}...')
print(f'  Messages: {len(result["messages"])}')

## 2. Writer Subgraph

In [ ]:
def mock_writer(topic: str, findings: str) -> str:
    return f'''Executive Summary: {topic}

Key Findings:
{findings[:200]}

Strategic Recommendation:
Based on research, we recommend immediate investment in AI capabilities.
ROI projected at 3-5x within 24 months.

Next Steps:
1. Establish AI Centre of Excellence
2. Partner with IIT/IIIT research labs
3. Pilot 3 AI use cases by Q2 2025'''

writer_graph = build_writer_subgraph(mock_writer)

result = writer_graph.invoke({
    'messages': [],
    'topic': 'India AI market',
    'findings': 'AI market growing 25% YoY. Govt investing $1.25B.',
    'draft': None
})

print('Writer subgraph result:')
print(result['draft'][:300])

## 3. Supervisor Pattern — Full Pipeline

The supervisor decides which agent to call next based on what has been done.
This is the most common multi-agent pattern in production systems.

In [ ]:
from day4.multi_agent import build_research_team

# Build team with custom functions
team = build_research_team(
    research_fn=mock_researcher,
    write_fn=mock_writer
)

result = team.invoke({
    'messages':     [HumanMessage('Research the India AI market for our board presentation')],
    'task':         'India AI market analysis for board deck',
    'next_agent':   None,
    'research':     None,
    'draft':        None,
    'final_output': None,
})

print(f'Task complete!')
print(f'Total agent messages: {len(result["messages"])}')
print()
print('Agent conversation:')
for msg in result['messages']:
    print(f'  {msg.content[:90]}')

In [ ]:
# Final output
print('FINAL OUTPUT')
print('=' * 60)
print(result['final_output'])

## 4. Multi-Agent Architecture Patterns

In [ ]:
patterns = {
    'Supervisor': {
        'description': 'One routing agent delegates to specialised sub-agents',
        'use_case': 'Customer service: triage -> billing agent OR tech agent OR returns agent',
        'companies': ['Salesforce Agentforce', 'ServiceNow', 'Freshworks']
    },
    'Subgraph': {
        'description': 'Independent graph used as a node inside parent graph',
        'use_case': 'Research pipeline: parent graph calls research subgraph as one step',
        'companies': ['McKinsey QuantumBlack', 'BCG Gamma', 'Deloitte AI']
    },
    'Handoff': {
        'description': 'Agent A explicitly routes to Agent B via state update',
        'use_case': 'Triage agent identifies complex case, hands off to specialist agent',
        'companies': ['Zendesk', 'Intercom', 'HubSpot']
    },
    'Parallel': {
        'description': 'Fan-out to N agents simultaneously, fan-in to combine',
        'use_case': 'Research 5 markets in parallel, combine findings in one summary',
        'companies': ['Bloomberg', 'Reuters', 'Thomson Reuters Westlaw']
    },
    'Reflection': {
        'description': 'Agent critiques own output, revises until quality threshold',
        'use_case': 'Code generator writes code, critic finds bugs, generator fixes them',
        'companies': ['GitHub Copilot', 'Cursor', 'Replit Ghostwriter']
    },
}

for name, info in patterns.items():
    print(f'{name}')
    print(f'  Description: {info["description"]}')
    print(f'  Use case:    {info["use_case"]}')
    print()

## Databricks Bridge

In [ ]:
# Multi-agent on Databricks:
#
# Option 1: Each agent writes to its own Delta table, supervisor reads them
# Option 2: Use Databricks Workflows to orchestrate agent tasks as jobs
# Option 3: MLflow to track each agent's runs and compare outputs
#
# import mlflow
#
# with mlflow.start_run(run_name='research_team'):
#     result = team.invoke({...})
#     mlflow.log_param('task', result['task'])
#     mlflow.log_metric('agent_messages', len(result['messages']))
#     mlflow.log_text(result['final_output'], 'final_output.txt')

print('Databricks: use MLflow to track multi-agent experiments and compare agent strategies.')